# House M.D. Chatbot Eğitimi

Bu notebook, mevcut tanı sınıflandırma modeline ek olarak arayüzde kullanılan chatbot modelini üretir. Chatbot, veri setindeki benzer vaka satırlarını TF-IDF ve en yakın komşu aramasıyla bulur; arayüz tarafında bu sonuçlar tanı modelinin tahminleriyle birlikte cevap metnine dönüştürülür.

Çıktı dosyası: `models/housemd_chatbot_retrieval_model.joblib`

In [ ]:
from __future__ import annotations

import re
import unicodedata
from datetime import datetime
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

ROOT = Path('.').resolve()
DATA_PATH = ROOT / 'DATA' / 'Last_HouseMD_DataSet.csv'
MODEL_PATH = ROOT / 'models' / 'housemd_chatbot_retrieval_model.joblib'

TARGET_COLUMN = 'correct_prediction'
TEXT_FEATURE_COLUMNS = ['text', 'Symptom', 'Test', 'Drug', 'Procedure', 'Organ']
META_FEATURE_COLUMNS = ['speaker', 'Intent', 'diagnosis_stage', 'Emotion', 'Sarcasm']
CASE_CONTEXT_TOKEN_LIMIT = 320

## Yardımcı Fonksiyonlar

Metin temizleme ve vaka bağlamı üretimi, arayüzdeki tahmin akışıyla aynı mantığı izler. Böylece chatbot araması ile sınıflandırma modeli benzer metin temsilleri üzerinde çalışır.

In [ ]:
def normalize_text(value):
    if value is None:
        return ''
    text = unicodedata.normalize('NFKC', str(value)).lower().replace('\u0307', '')
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^\w\s\-/+%.]', ' ', text, flags=re.UNICODE)
    text = text.replace('_', ' ')
    return re.sub(r'\s+', ' ', text).strip()


def extract_medical_entities(value):
    if value is None or not str(value).strip():
        return ''
    raw = str(value).strip().replace('""', '"')
    tokens = []
    entity_texts = re.findall(r'"text"\s*:\s*"([^"]+)"', raw)
    entity_types = re.findall(r'"type"\s*:\s*"([^"]+)"', raw)
    tokens.extend(normalize_text(text) for text in entity_texts)
    tokens.extend('entity_' + normalize_text(kind).replace(' ', '_') for kind in entity_types)
    return ' '.join(token for token in tokens if token)


def unique_token_join(values, limit=CASE_CONTEXT_TOKEN_LIMIT):
    seen = []
    used = set()
    for value in values:
        for token in str(value).split():
            if token and token not in used:
                seen.append(token)
                used.add(token)
            if len(seen) >= limit:
                return ' '.join(seen)
    return ' '.join(seen)


def build_row_text(row):
    parts = []
    for column in TEXT_FEATURE_COLUMNS:
        value = normalize_text(row.get(column, ''))
        if value:
            parts.append(value)
    for column in META_FEATURE_COLUMNS:
        value = normalize_text(row.get(column, ''))
        if value:
            parts.append(f'{column.lower()}_{value.replace(" ", "_")}')
    entities = extract_medical_entities(row.get('medical_entities', ''))
    if entities:
        parts.append(entities)
    return ' '.join(parts)


def build_case_context(row):
    fact_values = [row.get(column, '') for column in ['Symptom', 'Test', 'Drug', 'Procedure', 'Organ']]
    fact_text = unique_token_join([normalize_text(value) for value in fact_values])
    entity_text = unique_token_join([extract_medical_entities(row.get('medical_entities', ''))])
    return f'{fact_text} {entity_text}'.strip()


def compact(value, limit=180):
    text = str(value or '').strip()
    if len(text) <= limit:
        return text
    return text[: limit - 1].rstrip() + '...'

## Veriyi Hazırlama

In [ ]:
raw_df = pd.read_csv(DATA_PATH, sep=';', dtype=str).fillna('')
df = raw_df.copy()
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip()
df = df[df[TARGET_COLUMN].ne('') & df[TARGET_COLUMN].str.lower().ne('nan')].copy()

df['row_text'] = df.apply(build_row_text, axis=1)
df['case_context'] = df.apply(build_case_context, axis=1)
df['chat_text'] = (df['row_text'] + ' vaka_baglam ' + df['case_context']).str.strip()
df = df[df['chat_text'].str.len().gt(0)].copy()

df[[TARGET_COLUMN, 'text', 'chat_text']].head()

## TF-IDF + En Yakın Komşu Modeli

Bu model bir cevap üretici dil modeli değildir. Amacı, kullanıcının yazdığı vaka metnine en çok benzeyen House M.D. veri satırlarını hızlıca bulmaktır.

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=60000,
    min_df=1,
    sublinear_tf=True,
    norm='l2',
)
chat_matrix = vectorizer.fit_transform(df['chat_text'])

neighbors = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=min(6, len(df)))
neighbors.fit(chat_matrix)

chat_matrix.shape

## Arayüz İçin Kayıt Paketi

In [ ]:
records = []
for _, row in df.iterrows():
    records.append({
        'season': compact(row.get('season', ''), 20),
        'episode': compact(row.get('episode', ''), 20),
        'speaker': compact(row.get('speaker', ''), 80),
        'text': compact(row.get('text', ''), 260),
        'symptom': compact(row.get('Symptom', ''), 120),
        'test': compact(row.get('Test', ''), 120),
        'drug': compact(row.get('Drug', ''), 120),
        'procedure': compact(row.get('Procedure', ''), 120),
        'organ': compact(row.get('Organ', ''), 120),
        'intent': compact(row.get('Intent', ''), 80),
        'diagnosis_stage': compact(row.get('diagnosis_stage', ''), 80),
        'emotion': compact(row.get('Emotion', ''), 80),
        'label': compact(row.get(TARGET_COLUMN, ''), 160),
        'chat_text': compact(row.get('chat_text', ''), 420),
    })

package = {
    'model_type': 'tfidf_nearest_neighbor_chatbot',
    'version': '1.0',
    'saved_at': datetime.now().isoformat(timespec='seconds'),
    'data_path': str(DATA_PATH.relative_to(ROOT)),
    'vectorizer': vectorizer,
    'neighbors': neighbors,
    'records': records,
    'summary': {
        'rows': int(len(df)),
        'raw_rows': int(len(raw_df)),
        'class_count': int(df[TARGET_COLUMN].nunique()),
        'feature_count': int(chat_matrix.shape[1]),
    },
    'prompt_suggestions': [
        'Hasta nöbet geçiriyor ve MR sonucunda beyinde lezyon görülüyor. Ne düşünmeliyiz?',
        'Ateş, öksürük ve akciğer bulguları olan vaka için hangi tanı öne çıkar?',
        'Böbrek yetmezliği ve ilaç kullanımı birlikte görülüyorsa benzer vakalar ne söylüyor?',
    ],
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(package, MODEL_PATH, compress=3)
MODEL_PATH

## Hızlı Deneme

In [ ]:
query = 'Hasta nöbet geçiriyor ve MR sonucunda beyinde lezyon görülüyor.'
query_vector = vectorizer.transform([normalize_text(query)])
distances, indices = neighbors.kneighbors(query_vector, n_neighbors=3)

for distance, index in zip(distances[0], indices[0]):
    record = records[int(index)]
    print(round((1 - float(distance)) * 100, 2), record['label'], record['text'])